# Lesson 7.2: ANOVA: Comparing Three or More Means

Compare the exam scores of three teaching methods with a one-way ANOVA, then use Tukey's HSD to find which pairs of methods differ.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

## The question

An instructor taught three groups of five students with three methods (lecture, flipped classroom, problem-based), then gave all 15 the same exam. The group means are 80.0, 88.8 and 71.0. Do the teaching methods really differ, or is this the ordinary scatter you get between any 15 students? If they differ, which methods differ from which?

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## Step 1: Enter the three groups

Five students per teaching method, the same 15 scores as the lesson. We look at each group mean first, because ANOVA asks whether those means are further apart than the spread inside each group would explain.

In [1]:
import numpy as np  # Load NumPy for the numbers, here to get means and standard deviations.
from scipy import stats  # Load SciPy's statistics tools, which include the one-way ANOVA.
from statsmodels.stats.multicomp import pairwise_tukeyhsd  # Load Tukey's HSD from statsmodels, the follow-up test that compares every pair of groups.

lecture = [78, 82, 85, 79, 76]  # Exam scores of the five students taught by lecture.
flipped = [88, 91, 87, 93, 85]  # Scores of the five students in the flipped classroom.
problem = [72, 68, 74, 70, 71]  # Scores of the five students in the problem-based class.

for name, g in [('lecture', lecture), ('flipped', flipped), ('problem', problem)]:  # Loop over (label, scores) pairs so we write the summary once for all three groups.
    print(f'{name:8s} mean = {np.mean(g):.1f}   sd = {np.std(g, ddof=1):.2f}')  # Print each group's mean and its sample SD (ddof=1 divides by n - 1). ANOVA asks whether the means are further apart than this spread allows.

lecture  mean = 80.0   sd = 3.54
flipped  mean = 88.8   sd = 3.19
problem  mean = 71.0   sd = 2.24


**What this means.** Lecture averages 80.0, flipped 88.8 and problem-based 71.0. The SDs inside each group are small (2.24 to 3.54) compared with the 8 to 18 point gaps between the means, which hints that the methods differ.

## Step 2: Run the one-way ANOVA

f_oneway compares the spread between group means with the spread inside the groups. F near 1 means no difference; a large F with a small p-value says at least one mean differs.

In [2]:
res = stats.f_oneway(lecture, flipped, problem)  # f_oneway runs the one-way ANOVA: it compares the spread between the group means with the spread inside the groups.

print(f'F       = {res.statistic:.2f}')  # F is the ratio of those two spreads. Near 1 means no difference; larger means the means are further apart than chance explains.
print('df      = (2, 12)')  # Degrees of freedom: 3 groups - 1 = 2 between groups, and 15 - 3 = 12 within groups.
print(f'p-value = {res.pvalue:.7f}')  # The p-value is how rare an F this large would be if all three methods really had the same mean.

F       = 42.90
df      = (2, 12)
p-value = 0.0000034


**What this means.** F = 42.90 with 2 and 12 df gives p = 0.0000034. If the three methods truly had the same mean, an F this large would almost never happen, so at least one method differs from the others. It does not yet say which one.

## Step 3: Measure the effect size

Eta squared is the share of all the score variation that is explained by teaching method. It is the between-group sum of squares divided by the total.

In [3]:
scores = np.array(lecture + flipped + problem)  # Put all 15 scores in one array so we can find the overall mean and total variation.
grand = scores.mean()  # The grand mean is the average of all 15 scores. Every sum of squares is measured from it.

ssb = sum(len(g) * (np.mean(g) - grand) ** 2 for g in (lecture, flipped, problem))  # Between-group sum of squares: each group's mean distance from the grand mean, squared, times its size.
sst = ((scores - grand) ** 2).sum()  # Total sum of squares: how far all 15 scores are from the grand mean, squared and added up.

print(f'between SS = {ssb:.1f}, total SS = {sst:.1f}')  # Print both sums of squares so we can see how much of the total is between groups.
print(f'eta squared = {ssb / sst:.3f}')  # Eta squared = between / total, the share of all score variation explained by teaching method.

between SS = 792.1, total SS = 902.9
eta squared = 0.877


**What this means.** Between-group SS is 792.1 out of a total of 902.9, so eta squared = 0.877. About 88% of the variation in scores goes with teaching method, a very large effect. The rest is ordinary student-to-student scatter.

## Step 4: Find which pairs differ with Tukey

A significant ANOVA only says some mean is different. Tukey's HSD compares every pair while keeping the overall false-alarm rate at 5%. In the table, reject = True means that pair differs.

In [4]:
labels = ['lecture'] * 5 + ['flipped'] * 5 + ['problem'] * 5  # Make one label per score ('lecture' five times, and so on) in the same order as the scores.

tukey = pairwise_tukeyhsd(scores, labels, alpha=0.05)  # Run Tukey's HSD at alpha 0.05. It compares every pair of groups while holding the overall false-alarm rate at 5%.

print(tukey.summary())  # Print the table. meandiff is group2 minus group1, and reject = True means that pair really differs.

  Multiple Comparison of Means - Tukey HSD, FWER=0.05   
 group1  group2 meandiff p-adj   lower    upper   reject
--------------------------------------------------------
flipped lecture     -8.8 0.0017 -13.9271  -3.6729   True
flipped problem    -17.8    0.0 -22.9271 -12.6729   True
lecture problem     -9.0 0.0014 -14.1271  -3.8729   True
--------------------------------------------------------


**What this means.** Every pair differs. Lecture scored 8.8 below flipped (p = 0.0017), problem-based 17.8 below flipped (p < 0.0001) and 9.0 below lecture (p = 0.0014). So flipped is best, lecture is in the middle, and problem-based is lowest on this exam.

## So, what is the answer?

Yes, the teaching methods differ. The flipped classroom averaged 88.8, lecture 80.0 and problem-based 71.0, and every pair differs (ANOVA F = 42.90, p < 0.0001; eta squared = 0.88). Caution: with only five students per method, and no random assignment mentioned, this does not prove that the method itself caused the differences.

## Now you try

A gardener grows tomato plants with three fertilisers (A, B and C), six plants each, and records the total fruit weight per plant in kilograms. Do the fertilisers produce different yields, and if so, which ones differ?

**Data:** a = [21, 24, 19, 23, 22, 20]
b = [27, 30, 26, 29, 31, 28]
c = [22, 25, 24, 21, 26, 23]

1. Find the mean yield for each fertiliser.
2. Run a one-way ANOVA and report F and the p-value.
3. Compute eta squared.
4. Use Tukey's HSD to say which pairs of fertilisers differ.

*Hint:* Use stats.f_oneway(a, b, c), then pairwise_tukeyhsd on all 18 values with a matching list of labels. Eta squared is between SS divided by total SS.

In [ ]:
# Your turn: write your code here


### Solution (try first, then peek)

In [ ]:
import numpy as np
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

a = [21, 24, 19, 23, 22, 20]
b = [27, 30, 26, 29, 31, 28]
c = [22, 25, 24, 21, 26, 23]
res = stats.f_oneway(a, b, c)
print(f'F = {res.statistic:.2f}, p = {res.pvalue:.6f}')
y = np.array(a + b + c)
print('means:', [round(float(np.mean(x)), 1) for x in (a, b, c)])
print(pairwise_tukeyhsd(y, ['A'] * 6 + ['B'] * 6 + ['C'] * 6).summary())
ssb = sum(len(x) * (np.mean(x) - y.mean()) ** 2 for x in (a, b, c))
print(f'eta squared = {ssb / ((y - y.mean()) ** 2).sum():.3f}')

F = 22.29, p = 0.000032
means: [21.5, 28.5, 23.5]
Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower   upper  reject
----------------------------------------------------
     A      B      7.0    0.0  4.1944  9.8056   True
     A      C      2.0 0.1871 -0.8056  4.8056  False
     B      C     -5.0 0.0009 -7.8056 -2.1944   True
----------------------------------------------------
eta squared = 0.748


**Answer.** Means are A = 21.5, B = 28.5 and C = 23.5 kg. ANOVA gives F = 22.29, p = 0.000032, so the fertilisers do not all give the same yield, and eta squared = 0.748 (about 75% of the variation goes with fertiliser). Tukey: B beats A by 7.0 (p < 0.001) and beats C by 5.0 (p = 0.0009), but A and C do not differ (p = 0.187). So B is the best of these.

## The same question in R, SPSS and Stata

**R**

```r
score <- c(78, 82, 85, 79, 76, 88, 91, 87, 93, 85, 72, 68, 74, 70, 71)  # Enter the 15 scores in order: five lecture, then five flipped, then five problem-based.
method <- factor(rep(c("lecture", "flipped", "problem"), each = 5))  # Make a grouping variable (factor) that repeats each method name five times, in the same order as the scores.
fit <- aov(score ~ method)  # aov fits the one-way ANOVA. The formula score ~ method reads 'score explained by method'.
summary(fit)  # summary prints the ANOVA table with the F value and its p-value.
TukeyHSD(fit)  # TukeyHSD compares every pair of methods and adjusts the p-values so the overall false-alarm rate stays at 5%.
```

How to read it: In the ANOVA table, read F value 42.9 and Pr(>F) 3.41e-06 (that is 0.00000341): the methods differ. In the Tukey table, diff is one group minus the other and p adj below 0.05 means that pair differs: all three pairs do (-8.8, -17.8, -9.0).

**SPSS** (Analyze > Compare Means > One-Way ANOVA (Post Hoc: Tukey))

```
DATA LIST FREE / method score.
BEGIN DATA
1 78  1 82  1 85  1 79  1 76
2 88  2 91  2 87  2 93  2 85
3 72  3 68  3 74  3 70  3 71
END DATA.
ONEWAY score BY method
  /POSTHOC=TUKEY ALPHA(0.05).
```

How to read it: In the ANOVA table read F = 42.900 and Sig. = 0.000: the methods differ. In Multiple Comparisons (Tukey HSD) read Mean Difference and Sig. for each pair. All three Sig. values are below 0.05, so every pair differs. 1 = lecture, 2 = flipped, 3 = problem-based.

**Stata**

```
clear
set obs 15
generate score = real(word("78 82 85 79 76 88 91 87 93 85 72 68 74 70 71", _n))
generate method = ceil(_n / 5)
oneway score method
quietly anova score method
pwcompare method, mcompare(tukey) effects
```

How to read it: In the oneway table find F = 42.90 and Prob > F = 0.0000: the methods differ. In the pwcompare table, the Contrast column is the difference in means (for example 2 vs 1 is 8.8) and the Tukey p-value column shows each pair is below 0.05.